In [1]:
!nvidia-smi


Thu Oct 15 01:46:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00


In [3]:
import torch, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

2.11.0+cu130 True Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
ids = tok("What is the capital of France?")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

ids = tok("Quantization of transformers is unbelievably useful")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))

[3838, 374, 279, 6722, 315, 9625, 30]
['What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?']
What is the capital of France?
[44220, 2022, 315, 86870, 374, 38937, 88134, 5390]
['Quant', 'ization', 'Ġof', 'Ġtransformers', 'Ġis', 'Ġunbelie', 'vably', 'Ġuseful']


In [5]:
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
!nvidia-smi

Thu Oct 15 01:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             26W /   70W |    3239MiB /  15360MiB |     23%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
def bench(model, device, prompts, max_new_tokens=64, runs=3):
    inputs = tok(prompts, return_tensors="pt", padding=True).to(device)
    gen = dict(max_new_tokens=max_new_tokens, min_new_tokens=max_new_tokens, do_sample=False)
    model.generate(**inputs, **{**gen, "max_new_tokens": 8, "min_new_tokens": 8})  # warmup
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(runs):
        if device == "cuda":
            torch.cuda.synchronize()
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            s.record()
            out = model.generate(**inputs, **gen)
            e.record(); torch.cuda.synchronize()
            times.append(s.elapsed_time(e) / 1000)
        else:
            t0 = time.perf_counter()
            out = model.generate(**inputs, **gen)
            times.append(time.perf_counter() - t0)
    avg = sum(times) / len(times)
    new_tokens = max_new_tokens * len(prompts)
    mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else None
    return {"latency_s": avg, "tokens_per_s": new_tokens / avg, "peak_mem_gb": mem}


In [8]:
test_prompts = ["What is the capital of France?"]
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'latency_s': 2.59376171875, 'tokens_per_s': 12.337293656805766, 'peak_mem_gb': 3.104342528}


In [9]:
print(bench(model, "cuda", test_prompts, max_new_tokens=32))
print(bench(model, "cuda", test_prompts, max_new_tokens=32))

{'latency_s': 2.2211682942708335, 'tokens_per_s': 14.406832693650069, 'peak_mem_gb': 3.104342528}
{'latency_s': 2.8682935384114585, 'tokens_per_s': 11.156459257556497, 'peak_mem_gb': 3.104342528}


Baseline, batch size 1, fp16, 32 new tokens, Tesla T4
Session 1: 2.31 s (13.9 tok/s, slow first call), 1.24 s (25.8), 1.26 s (25.4)
Session 2: 1.27 s (25.2), 1.28 s (25.0)
Peak memory: 3.10 GB
Note: the first call in session 1 was ~2x slower and didn't reproduce, probably a one-time cold start.

In [10]:
prompts = [
    "What is the capital of France?",
    "Explain what a GPU is in two sentences.",
    "Write a Python function that returns the nth Fibonacci number.",
    "List three differences between a CPU and a GPU.",
    "If a train travels 60 miles in 1.5 hours, what is its average speed?",
    "Translate to Spanish: Where is the nearest train station?",
    "Give me three tips for preparing for a technical interview.",
    "What does the word 'latency' mean in computing?",
    "Write a haiku about the ocean.",
    "Explain what quantization means for a neural network in simple terms.",
    "What is 17 multiplied by 23?",
    "Write a short, polite email asking a professor for an extension.",
    "Name three uses of machine learning in healthcare.",
    "What is the difference between a list and a tuple in Python?",
    "Describe how a CUDA thread block works in one paragraph.",
    "Why is the sky blue?",
]
print(len(prompts))

enc = tok(prompts[:4], return_tensors="pt", padding=True)
print(enc["input_ids"].shape)
print(enc["attention_mask"])

16
torch.Size([4, 11])
tensor([[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [11]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()

In [12]:
!nvidia-smi

Thu Oct 15 01:56:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             28W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
cpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)  # stays on CPU
print("CPU threads:", torch.get_num_threads())
cpu_result = bench(cpu_model, "cpu", prompts[:1], max_new_tokens=32, runs=2)
print("CPU fp32:", cpu_result)
del cpu_model
gc.collect()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

CPU threads: 1
CPU fp32: {'latency_s': 18.306682123500025, 'tokens_per_s': 1.7479956107896832, 'peak_mem_gb': None}


2747

In [14]:
gpu_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda")
gpu_result = bench(gpu_model, "cuda", prompts[:1], max_new_tokens=32, runs=3)
print("GPU fp32:", gpu_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GPU fp32: {'latency_s': 1.183460693359375, 'tokens_per_s': 27.039343325518235, 'peak_mem_gb': 6.192115712}


In [15]:
print("GPU fp32 vs CPU fp32 speedup:", gpu_result["tokens_per_s"] / cpu_result["tokens_per_s"])

GPU fp32 vs CPU fp32 speedup: 15.468770721513886


In [19]:
# del gpu_model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [20]:
from torch.profiler import profile, ProfilerActivity

inputs = tok(prompts[:1], return_tensors="pt").to("cuda")

for _ in range(2):
    model.generate(**inputs, max_new_tokens=8, do_sample=False)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    model.generate(**inputs, max_new_tokens=16, do_sample=False)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         1.79%      15.319ms        22.23%     190.637ms      60.481us       0.000us         0.00%     218.222ms      69.233us          3152  
                                           aten::matmul         2.45%      20.981ms        17.55%     150.506ms      55.333us       0.000us         0.00%     204.225ms      75.083us          2720  
         